# Query AR vs causal encoder on the S=7 sweep (read-only plots)

Run this after notebooks 29a / 29b (query_ar, seed 42) have finished. It reads saved predictions only: no
training and no re-evaluation, so a CPU runtime is enough.

**Models compared by default.**
* `query_ar` (notebooks 29a/29b): identical to the causal encoder except that output position j also receives the
  digit of cell j−1, and it decodes cell by cell.
* `causal_encoder` (notebooks 23a/23b, and 27a/27b for extra seeds): 36 output queries, all predicted in one pass.

Both use the same test records. The default seed is 42, the only seed query_ar has so far. Add `"transformer_ar"`
to `MODELS` to also show the original AR.

**Plots.**
1. One panel per training support (k02 … k0234567): exact accuracy vs test k (k=1 = singletons). Grey = trained k,
   open markers = held out, dashed line = longest trained k.
2. Singleton / +1 step / +2 step accuracy vs longest trained k.
3. Tables: accuracy by condition, model and k; and a compact held-out summary.

**Send back `QueryAR_vs_CausalEncoder_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = "full"
SEEDS = [42]                                         # query_ar has seed 42 only (so far)
MODELS = ["query_ar", "causal_encoder"]              # add "transformer_ar" to include the original AR
SUBDIR = "S7_support_sweep_20k_v1"
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX


In [ ]:
# --- Plots + tables (reads saved artifacts only) ---------------------------------------------------------------------
rp = LX.multiseed_report(RESULTS_ROOT, MODE, SEEDS, LK.S7_SWEEP, SUBDIR, models=MODELS,
                         out_name="query_ar_vs_causal_encoder", review_name="QueryAR_vs_CausalEncoder_Review.ipynb")
df = LX.per_k_accuracy(RESULTS_ROOT, MODE, SEEDS, LK.S7_SWEEP, SUBDIR, MODELS)
ok = df[df.status == "available"]
held = ok[ok.role != "trained"]
summary = held.pivot_table(index=["condition", "model"], columns="role", values="exact_acc", aggfunc="mean").round(4)
summary["trained k (min)"] = ok[ok.role == "trained"].groupby(["condition", "model"]).exact_acc.min().round(4)
cols = [c for c in ["trained k (min)", "singleton", "+1 step", "+2 step", "+3 step", "+4 step", "+5 step"] if c in summary.columns]
print("Held-out summary (mean over SEEDS):")
display(summary[cols].reset_index())
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))
